In [1]:
%pip install --quiet pandas numpy openpyxl textblob


Note: you may need to restart the kernel to use updated packages.


In [7]:
import os, re, math, json
import numpy as np
import pandas as pd

# ====== CONFIG ======
# Point this to the CSV produced by your batch OCR step
CSV_INPUT = r"C:\Users\guest441\Downloads\Lishebora_Version_2\Lishebora_Version_2\Ingredients\ingredients_extracted_all.csv"

# Where to save the Excel output
XLSX_OUTPUT = r"C:\Users\guest441\Downloads\Lishebora_Version_2\Lishebora_Version_2\Ingredients\ingredients_postprocessed.xlsx"

# Toggle spelling correction (slower). If True, requires TextBlob.
USE_SPELLCHECK = False

# Canonical replacements (extend freely)
CANON_MAP = {
    "suger": "sugar",
    "sugr": "sugar",
    "salt.": "salt",
    "flovour": "flavour",
    "flavor": "flavour",   # pick UK/US you prefer
    "colour": "color",
    "colur": "color",
    "citric accid": "citric acid",
    "wheet": "wheat",
    "maize flour": "corn flour",
}

# Base unit map (mass->g, volume->ml, spoons/cups approximated)
UNIT_PATTERNS = {
    "kg": ("g", 1000.0), "g": ("g", 1.0), "mg": ("g", 0.001),
    "l": ("ml", 1000.0), "lt": ("ml", 1000.0), "ml": ("ml", 1.0),
    "tbsp": ("ml", 15.0), "tablespoon": ("ml", 15.0),
    "tsp": ("ml", 5.0), "teaspoon": ("ml", 5.0),
    "cup": ("ml", 240.0), "cups": ("ml", 240.0),
}

# Quantities like: 12g, 12 g, 0.5 L, 3 tbsp, 2%, 1/2 tsp, ½ tsp
QTY_REGEX = re.compile(
    r"""
    (?P<num>
        (?:\d+(?:\.\d+)?)         # 12 or 12.5
        |(?:\d+\s*/\s*\d+)        # 1/2
        |(?:[¼½¾⅓⅔⅛⅜⅝⅞])         # unicode fractions
    )
    \s*
    (?P<unit>
        %(?!\w)                   # % as unit
        |kg|g|mg|l|lt|ml|tsp|teaspoon|tbsp|tablespoon|cup|cups
    )?
    """, re.IGNORECASE | re.VERBOSE
)

UNICODE_FRAC = {
    "¼": 0.25, "½": 0.5, "¾": 0.75,
    "⅓": 1/3,  "⅔": 2/3,
    "⅛": 0.125,"⅜": 0.375,"⅝": 0.625,"⅞": 0.875,
}

# ---- optional spell checker ----
Word = None
if USE_SPELLCHECK:
    try:
        from textblob import Word as TBWord
        Word = TBWord
    except Exception as e:
        print("[WARN] TextBlob unavailable; continuing without spellcheck.", e)
        USE_SPELLCHECK = False

def parse_number_token(tok: str) -> float | None:
    tok = tok.strip()
    if tok in UNICODE_FRAC:
        return UNICODE_FRAC[tok]
    if "/" in tok:
        try:
            a, b = tok.split("/")
            return float(a) / float(b)
        except:
            return None
    try:
        return float(tok)
    except:
        return None

def normalize_unit(u: str | None) -> str | None:
    if not u: return None
    u = u.lower()
    return "%" if u == "%" else u

def convert_to_base(num: float, unit: str | None):
    """Return (value, base_unit, kind) where kind in {'mass','volume','percent','unknown'}."""
    if unit == "%" or unit == "percent":
        return (num, "%", "percent")
    if unit and unit in UNIT_PATTERNS:
        base_unit, factor = UNIT_PATTERNS[unit]
        return (num * factor, base_unit, "mass" if base_unit == "g" else "volume")
    return (num, unit, "unknown")

def clean_ingredient_name(name: str) -> str:
    base = name.strip().lower()
    base = re.sub(r"^[\s\-\•\·\●\:]+", "", base)       # leading bullets/punct
    base = re.sub(r"[\.\;\:]+$", "", base)             # trailing punct
    base = re.sub(r"\s+", " ", base).strip()
    base = CANON_MAP.get(base, base)
    if USE_SPELLCHECK and Word is not None:
        parts = []
        for w in re.findall(r"[A-Za-z]+|[0-9\%\-\+\.\(\)]+|\s+", base):
            if w.isalpha() and len(w) > 2:
                parts.append(str(Word(w).correct()))
            else:
                parts.append(w)
        base = "".join(parts)
        base = re.sub(r"\s+", " ", base).strip()
    base = CANON_MAP.get(base, base)
    exceptions = {"of","and","with","in","on","to","for"}
    return " ".join([w.capitalize() if w not in exceptions else w for w in base.split()])

def extract_qty_from_text(text: str):
    """Return list of {value, unit, base_value, base_unit, kind} found in text."""
    out = []
    for m in QTY_REGEX.finditer(text):
        num = parse_number_token(m.group("num"))
        unit = normalize_unit(m.group("unit"))
        if num is None:
            continue
        base_val, base_unit, kind = convert_to_base(num, unit)
        out.append({
            "value": num,
            "unit": unit or "",
            "base_value": base_val,
            "base_unit": base_unit or "",
            "kind": kind
        })
    return out

def guess_net_quantity(raw_text: str):
    """Detect total package size (best-effort) from OCR raw_text."""
    patterns = [
        r"(net\s*(?:wt|weight)[:\s]*)(?P<num>\d+(?:\.\d+)?)[\s]*(?P<unit>kg|g|mg|l|lt|ml)\b",
        r"(net[:\s]*)(?P<num>\d+(?:\.\d+)?)[\s]*(?P<unit>kg|g|mg|l|lt|ml)\b",
        r"(?P<num>\d+(?:\.\d+)?)[\s]*(?P<unit>kg|g|mg|l|lt|ml)\b\s*(?:net)?",
    ]
    text = (raw_text or "").lower()
    for p in patterns:
        m = re.search(p, text)
        if m:
            num = float(m.group("num"))
            unit = m.group("unit")
            base_val, base_unit, kind = convert_to_base(num, unit)
            return {"value": num, "unit": unit, "base_value": base_val, "base_unit": base_unit, "kind": kind}
    return None


In [8]:
# SAFE SAVE: recompute tidy/summary if missing, then write Excel
import os, numpy as np, pandas as pd

# 0) Make sure the output folder exists
os.makedirs(os.path.dirname(XLSX_OUTPUT), exist_ok=True)

def _recompute_from_csv():
    # uses helpers you already defined: clean_ingredient_name, extract_qty_from_text, guess_net_quantity
    if not os.path.exists(CSV_INPUT):
        raise FileNotFoundError(f"CSV_INPUT not found: {CSV_INPUT}")

    df = pd.read_csv(CSV_INPUT)
    for col in ["file", "ingredients", "raw_text"]:
        if col not in df.columns:
            raise ValueError(f"Missing required column in CSV: {col}")

    # explode to tidy rows
    df["ingredient_list"] = df["ingredients"].fillna("").astype(str).str.split(r";\s*")
    _tidy = df[["file", "raw_text", "ingredient_list"]].explode("ingredient_list", ignore_index=True)
    _tidy = _tidy.rename(columns={"ingredient_list": "ingredient_raw"})
    _tidy["ingredient_raw"] = _tidy["ingredient_raw"].fillna("").astype(str).str.strip()
    _tidy = _tidy[_tidy["ingredient_raw"] != ""]

    # clean names + extract first qty
    _tidy["ingredient_clean"] = _tidy["ingredient_raw"].apply(clean_ingredient_name)
    for c in ["qty_value","qty_unit","qty_base_value","qty_base_unit","qty_kind"]:
        _tidy[c] = np.nan

    for idx, row in _tidy.iterrows():
        qlist = extract_qty_from_text(row["ingredient_raw"])
        if qlist:
            q = qlist[0]
            _tidy.at[idx, "qty_value"] = q["value"]
            _tidy.at[idx, "qty_unit"] = q["unit"]
            _tidy.at[idx, "qty_base_value"] = q["base_value"]
            _tidy.at[idx, "qty_base_unit"] = q["base_unit"]
            _tidy.at[idx, "qty_kind"] = q["kind"]

    # per-product totals
    totals = []
    for file_path, sub in _tidy.groupby("file", dropna=False):
        raw_text_sample = sub["raw_text"].iloc[0] if not sub["raw_text"].isna().all() else ""
        net = guess_net_quantity(raw_text_sample or "")

        total_g  = sub.loc[(sub["qty_kind"]=="mass")   & sub["qty_base_value"].notna(), "qty_base_value"].sum()
        total_ml = sub.loc[(sub["qty_kind"]=="volume") & sub["qty_base_value"].notna(), "qty_base_value"].sum()
        total_pct= sub.loc[(sub["qty_kind"]=="percent")& sub["qty_value"].notna(),      "qty_value"].sum()

        totals.append({
            "file": file_path,
            "sum_ingredient_mass_g": float(total_g) if not np.isnan(total_g) else 0.0,
            "sum_ingredient_volume_ml": float(total_ml) if not np.isnan(total_ml) else 0.0,
            "sum_ingredient_percent": float(total_pct) if not np.isnan(total_pct) else 0.0,
            "net_detected_value": (net["base_value"] if net else np.nan),
            "net_detected_unit":  (net["base_unit"]  if net else ""),
            "net_detected_kind":  (net["kind"]       if net else ""),
        })

    _summary = (
        _tidy.assign(has_qty = _tidy["qty_kind"].notna())
             .groupby("file", as_index=False)
             .agg(n_ingredients=("ingredient_clean","count"),
                  n_with_qty=("has_qty","sum"))
             .merge(pd.DataFrame(totals), on="file", how="left")
    )

    return _tidy, _summary

# 1) Ensure tidy/summary exist
if 'tidy' not in globals() or 'summary' not in globals():
    print("tidy/summary not found in memory — recomputing from CSV_INPUT …")
    tidy, summary = _recompute_from_csv()

# 2) If still empty, create minimal frames so openpyxl writes at least one sheet
if tidy is None or len(tidy) == 0:
    print("[WARN] 'tidy' is empty; writing an empty placeholder sheet.")
    tidy = pd.DataFrame(columns=["file","raw_text","ingredient_raw","ingredient_clean",
                                 "qty_value","qty_unit","qty_base_value","qty_base_unit","qty_kind"])
if summary is None or len(summary) == 0:
    print("[WARN] 'summary' is empty; writing an empty placeholder sheet.")
    summary = pd.DataFrame(columns=["file","n_ingredients","n_with_qty",
                                    "sum_ingredient_mass_g","sum_ingredient_volume_ml","sum_ingredient_percent",
                                    "net_detected_value","net_detected_unit","net_detected_kind"])

# 3) Write Excel
with pd.ExcelWriter(XLSX_OUTPUT, engine="openpyxl") as w:
    tidy.sort_values(["file","ingredient_clean"], na_position="last").to_excel(
        w, sheet_name="ingredients_tidy", index=False
    )
    summary.to_excel(w, sheet_name="per_product_summary", index=False)

print("Saved Excel:", XLSX_OUTPUT)
print("Rows written — tidy:", len(tidy), "| summary:", len(summary))


tidy/summary not found in memory — recomputing from CSV_INPUT …


C:\Users\guest441\AppData\Local\Temp\ipykernel_8392\3110514302.py:34: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  _tidy.at[idx, "qty_unit"] = q["unit"]
C:\Users\guest441\AppData\Local\Temp\ipykernel_8392\3110514302.py:36: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  _tidy.at[idx, "qty_base_unit"] = q["base_unit"]
C:\Users\guest441\AppData\Local\Temp\ipykernel_8392\3110514302.py:37: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'unknown' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  _tidy.at[idx, "qty_kind"] = q["

Saved Excel: C:\Users\guest441\Downloads\Lishebora_Version_2\Lishebora_Version_2\Ingredients\ingredients_postprocessed.xlsx
Rows written — tidy: 1307 | summary: 166
